# BPSD — corrected pipeline (PPAC-comparable protocol)

Cells:
 1. Config + splits (PPAC-identical, incl. the balanced/intervened test set)
 2. LightGCN backbone (Stage 1)
 3. Behavioural proxies from *metadata*, not from the backbone (Stage 2)
 4. PPD popularity proxies (Stage 3)
 5. Behaviour-guided popularity subspace + training + eval (Stages 4-7)



In [1]:
# ============================================================================
# CELL 1 — CONFIG AND SPLITS
# ============================================================================
import os, sys, time, math, json, random, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

RAW_DATA_DIR = '/kaggle/input/datasets/tejasdeshmukh001/movielens'   # ratings.dat, movies.dat
WORKING_DIR  = '/kaggle/working/dataset/ml-1M'
CKPT_DIR     = '/kaggle/working/checkpoints'
os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 2020
POSITIVE_RATING_THRESHOLD = 4.0   # PPAC uses `rating > 3`
TEST_HOLDOUT_PER_USER     = 30    # PPAC's TOP_K
BALANCE_PER_ITEM          = None  # None -> derive it from your own holdout pool (see below)
VAL_HOLDOUT_PER_USER      = 20    # only used when STRICT_PPAC = False

# STRICT_PPAC = True  -> reproduce the paper's released protocol exactly:
#                        no validation split, checkpoint selected by NDCG@50 on the
#                        balanced test set (this is literally what run_MF.py does).
#                        Use this ONLY for the head-to-head number against Table 2.
# STRICT_PPAC = False -> carve a validation split and select on it. Honest, but the
#                        absolute numbers land a little below the paper's.
STRICT_PPAC = False

EVAL_ON_BALANCED = True   # True  -> balanced (intervened) test set  == paper's numbers
                          # False -> the raw 30-per-user holdout     == biased test set

TOP_KS = [20, 50, 100]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


def build_splits():
    """Reproduce PPAC's ml-1M split, plus its balanced (intervened) test set.

    PPAC (run_MF.py::create_train_and_test):
      * positives are ratings > 3
      * every user with MORE than 30 positives contributes exactly 30 to the test pool
      * every other user contributes all of their positives to train
      * NO global minimum-interaction user filter
      * NO validation split

    Balanced (intervened) test set: from the test pool, fix a per-item quota n,
    keep every item with at least n held-out interactions, and subsample exactly n
    of them. Every retained item then contributes the same number of test
    interactions, which is the property the protocol needs.

    n is not a free constant. Retained interactions = n * |{i : c_i >= n}|, which
    has an interior maximum: small n throws away interactions from popular items,
    large n throws away items entirely. BALANCE_PER_ITEM = None picks the argmax
    from THIS split's own item counts, so nothing is inherited from anywhere.

    Built entirely from ratings.dat and movies.dat. No external split files.
    """
    rng = random.Random(SEED)
    by_user = collections.defaultdict(list)
    rating_of = {}
    ts_of = {}

    with open(os.path.join(RAW_DATA_DIR, 'ratings.dat'), encoding='latin-1') as fh:
        for line in fh:
            u, i, r, t = line.strip().split('::')
            u, r, t = int(u), float(r), int(t)
            if r >= POSITIVE_RATING_THRESHOLD:
                by_user[u].append(i)
                rating_of[(u, i)] = r
                ts_of[(u, i)] = t

    train, val, test_pool = (collections.defaultdict(list) for _ in range(3))
    for u, items in by_user.items():
        items = list(items)
        rng.shuffle(items)
        need = TEST_HOLDOUT_PER_USER + (0 if STRICT_PPAC else VAL_HOLDOUT_PER_USER)
        if len(items) > need:
            test_pool[u] = items[:TEST_HOLDOUT_PER_USER]
            if not STRICT_PPAC:
                val[u] = items[TEST_HOLDOUT_PER_USER:need]
            train[u] = items[need:]
        else:
            train[u] = items

    # --- balanced / intervened sets -----------------------------------------
    def _balance(pool, quota=None):
        """Keep every item with >= n held-out interactions, subsample exactly n.

        n defaults to the argmax of retained(n) = n * |{i : c_i >= n}|, derived
        from this pool's own item counts. Applied to the validation pool as well
        as the test pool, so selection and reporting see the same kind of set.
        """
        hits = collections.defaultdict(list)
        for u, items in pool.items():
            for i in items:
                hits[i].append(u)
        counts = np.array(sorted(len(v) for v in hits.values()))
        retained = counts[::-1] * (np.arange(len(counts)) + 1)
        q = int(quota) if quota else int(counts[::-1][retained.argmax()])
        out, n_items = collections.defaultdict(list), 0
        for i, users in hits.items():
            if len(users) >= q:
                n_items += 1
                for u in rng.sample(users, q):
                    out[u].append(i)
        return out, q, n_items

    balanced, quota, n_bal_items = _balance(test_pool, BALANCE_PER_ITEM)

    print(f'[split] users={len(by_user)}  train_inter={sum(map(len, train.values()))}')
    print(f'[split] raw test: {len(test_pool)} users, {sum(map(len, test_pool.values()))} interactions')
    print(f'[split] balanced test: {len(balanced)} users, {sum(map(len, balanced.values()))} '
          f'interactions over {n_bal_items} items @ {quota} each')
    if not STRICT_PPAC:
        print(f'[split] val: {len(val)} users, {sum(map(len, val.values()))} interactions')

    def dump(name, recs):
        with open(os.path.join(WORKING_DIR, name), 'w', encoding='utf-8') as fh:
            for u, items in recs.items():
                for i in items:
                    fh.write(f'{u}::{i}::{rating_of[(u, i)]:.1f}::{ts_of[(u, i)]}\n')

    dump('ratings.train', train)
    dump('ratings.test', test_pool)
    dump('balance_ratings.test', balanced)
    if not STRICT_PPAC:
        dump('ratings.val', val)
        bal_val, vq, vn = _balance(val)
        print(f'[split] balanced val: {len(bal_val)} users, '
              f'{sum(map(len, bal_val.values()))} interactions over {vn} items @ {vq} each')
        dump('balance_ratings.val', bal_val)


def read_splits():
    """Load the splits and remap to compact 0-based ids."""
    item_map = {}
    with open(os.path.join(RAW_DATA_DIR, 'movies.dat'), encoding='latin-1') as fh:
        for idx, line in enumerate(fh):
            item_map[line.split('::')[0]] = idx

    def load(name):
        recs = collections.defaultdict(list)
        path = os.path.join(WORKING_DIR, name)
        if not os.path.exists(path):
            return recs
        with open(path, encoding='utf-8') as fh:
            for line in fh:
                u, i, _, _ = line.strip().split('::')
                recs[int(u)].append(item_map[i])
        return recs

    raw_train = load('ratings.train')
    user_map = {raw: new for new, raw in enumerate(sorted(raw_train))}
    train = collections.defaultdict(list, {user_map[u]: v for u, v in raw_train.items()})
    train_items = {i for v in train.values() for i in v}

    def rankable(name):
        out = collections.defaultdict(list)
        for u, items in load(name).items():
            if u not in user_map:
                continue
            keep = [i for i in items if i in train_items]
            if keep:
                out[user_map[u]] = keep
        return out

    test = rankable('balance_ratings.test' if EVAL_ON_BALANCED else 'ratings.test')
    val = rankable('balance_ratings.val' if EVAL_ON_BALANCED else 'ratings.val')
    if STRICT_PPAC:
        val = test          # paper protocol: selection happens on the eval set
    params = {'num_users': len(user_map), 'num_items': len(item_map)}
    return train, val, test, user_map, item_map, params


set_seed(SEED)
build_splits()
train_records, val_records, test_records, user_map, item_map, params = read_splits()
NUM_USERS, NUM_ITEMS = params['num_users'], params['num_items']
print(f'[data] users={NUM_USERS} items={NUM_ITEMS} '
      f'train={sum(map(len, train_records.values()))} '
      f'val_users={len(val_records)} test_users={len(test_records)}')

[split] users=6038  train_inter=408831
[split] raw test: 3329 users, 99870 interactions
[split] balanced test: 3329 users, 29882 interactions over 446 items @ 67 each
[split] val: 3329 users, 66580 interactions
[split] balanced val: 3325 users, 19932 interactions over 453 items @ 44 each
[data] users=6038 items=3883 train=408831 val_users=3325 test_users=3329


In [3]:
# ============================================================================
# CELL 2 — STAGE 1: LIGHTGCN BACKBONE
# ============================================================================
LATENT_DIM = 64
N_LAYERS   = 3
LR         = 1e-3       # paper reports 0.01; 1e-3 is the released-code default
REG_WEIGHT = 1e-4
BATCH_SIZE = 8192
EPOCHS     = 2000
PATIENCE   = 50


def build_sparse_adj(train_recs, num_users, num_items):
    n = num_users + num_items
    us, it = [], []
    for u, items in train_recs.items():
        us.extend([u] * len(items))
        it.extend([i + num_users for i in items])
    us = torch.tensor(us, dtype=torch.long)
    it = torch.tensor(it, dtype=torch.long)
    src = torch.cat([us, it]); dst = torch.cat([it, us])
    deg = torch.zeros(n).scatter_add_(0, dst, torch.ones(dst.numel()))
    dis = deg.clamp(min=1.0).pow(-0.5)
    return torch.sparse_coo_tensor(
        torch.stack([dst, src]), dis[dst] * dis[src], (n, n)).coalesce()


def propagate(adj, e_u0, e_i0, n_layers=N_LAYERS):
    """LightGCN readout: mean over layers 0..L (alpha_l = 1/(L+1))."""
    x = torch.cat([e_u0, e_i0], dim=0)
    layers = [x]
    for _ in range(n_layers):
        x = torch.sparse.mm(adj, x)
        layers.append(x)
    out = torch.stack(layers, dim=1).mean(dim=1)
    return torch.split(out, [e_u0.shape[0], e_i0.shape[0]], dim=0)


def sample_triplets(train_recs, num_items):
    users, pos = [], []
    for u, items in train_recs.items():
        users.extend([u] * len(items)); pos.extend(items)
    users = np.asarray(users, dtype=np.int64); pos = np.asarray(pos, dtype=np.int64)
    psets = {u: set(v) for u, v in train_recs.items()}
    neg = np.random.randint(0, num_items, size=len(users), dtype=np.int64)
    bad = np.fromiter((n in psets[u] for u, n in zip(users, neg)), bool, len(users))
    while bad.any():
        idx = np.flatnonzero(bad)
        neg[idx] = np.random.randint(0, num_items, size=len(idx), dtype=np.int64)
        bad[idx] = np.fromiter((neg[j] in psets[users[j]] for j in idx), bool, len(idx))
    return torch.from_numpy(users), torch.from_numpy(pos), torch.from_numpy(neg)


def recall_ndcg(ground_truth, ranked, k):
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    rec, ndcg = [], []
    for truth, row in zip(ground_truth, ranked):
        ts = set(truth)
        hits = np.fromiter((i in ts for i in row[:k]), np.float32, k)
        rec.append(hits.sum() / len(ts))
        idcg = disc[:min(len(ts), k)].sum()
        ndcg.append(float((hits * disc).sum()) / idcg if idcg > 0 else 0.0)
    return float(np.mean(rec)), float(np.mean(ndcg))


@torch.no_grad()
def evaluate(e_u, e_i, eval_recs, exclude_recs, item_pop, ks=TOP_KS):
    users = sorted(u for u, v in eval_recs.items() if v)
    max_k = max(ks)
    ranked_all = []
    for s in range(0, len(users), 512):
        batch = users[s:s + 512]
        scores = e_u[batch] @ e_i.T
        for r, u in enumerate(batch):
            ex = exclude_recs.get(u, [])
            if ex:
                scores[r, torch.tensor(ex, dtype=torch.long, device=scores.device)] = -torch.inf
        ranked_all.append(torch.topk(scores, k=max_k, dim=1).indices.cpu().numpy())
    ranked = np.concatenate(ranked_all, 0)
    truth = [eval_recs[u] for u in users]

    out = {}
    for k in ks:
        r, n = recall_ndcg(truth, ranked, k)
        out[k] = {'recall': r, 'ndcg': n, 'arp': float(item_pop[ranked[:, :k]].mean())}
    tail = set(np.flatnonzero(item_pop <= np.median(item_pop)).tolist())
    tt = [[i for i in t if i in tail] for t in truth]
    keep = [j for j, t in enumerate(tt) if t]
    out['tail_recall@50'] = recall_ndcg([tt[j] for j in keep], ranked[keep], 50)[0] if keep else 0.0
    return out


class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, dim=LATENT_DIM):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, dim)
        self.item_embedding = nn.Embedding(num_items, dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def readout(self, adj):
        return propagate(adj, self.user_embedding.weight, self.item_embedding.weight)


def item_popularity_from(train_recs, num_items):
    c = np.zeros(num_items, dtype=np.float32)
    for items in train_recs.values():
        c[np.asarray(items, dtype=np.int64)] += 1.0
    return c / max(float(c.max()), 1.0)


def train_backbone():
    set_seed(SEED)
    adj = build_sparse_adj(train_records, NUM_USERS, NUM_ITEMS).to(DEVICE)
    model = LightGCN(NUM_USERS, NUM_ITEMS).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    item_pop = item_popularity_from(train_records, NUM_ITEMS)
    ckpt = os.path.join(CKPT_DIR, 'lightgcn-ml1m.pt')

    best, best_ep = -np.inf, -1
    for epoch in range(EPOCHS):
        model.train(); t0 = time.time()
        u, p, n = sample_triplets(train_records, NUM_ITEMS)
        perm = torch.randperm(len(u)); u, p, n = u[perm], p[perm], n[perm]
        tot = 0.0; nb = 0
        for s in range(0, len(u), BATCH_SIZE):
            ub = u[s:s+BATCH_SIZE].to(DEVICE)
            pb = p[s:s+BATCH_SIZE].to(DEVICE)
            nb_ = n[s:s+BATCH_SIZE].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            eu, ei = model.readout(adj)
            pos = (eu[ub] * ei[pb]).sum(1)
            neg = (eu[ub] * ei[nb_]).sum(1)
            rank = F.softplus(neg - pos).mean()
            ego = (model.user_embedding(ub).pow(2).sum()
                   + model.item_embedding(pb).pow(2).sum()
                   + model.item_embedding(nb_).pow(2).sum()) / (2.0 * len(ub))
            loss = rank + REG_WEIGHT * ego
            loss.backward(); opt.step()
            tot += loss.item(); nb += 1

        model.eval()
        with torch.no_grad():
            eu, ei = model.readout(adj)
            m = evaluate(eu, ei, val_records, train_records, item_pop)
        if m[50]['ndcg'] > best:
            best, best_ep = m[50]['ndcg'], epoch
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, ckpt)
        print(f'Epoch [{epoch+1}/{EPOCHS}] Loss {tot/nb:.4f} '
              f'Recall@50 {m[50]["recall"]:.4f} NDCG@50 {m[50]["ndcg"]:.4f} '
              f'ARP@50 {m[50]["arp"]:.4f} {time.time()-t0:.1f}s')
        if epoch - best_ep >= PATIENCE:
            print(f'Early stop at {epoch+1}; best epoch {best_ep+1}'); break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)['model_state_dict'])
    torch.save({'adjacency': adj.cpu().coalesce(),
                'num_users': NUM_USERS, 'num_items': NUM_ITEMS},
               '/kaggle/working/sparse_adj_matrix.pt')
    model.eval()
    with torch.no_grad():
        eu, ei = model.readout(adj)
        excl = train_records if STRICT_PPAC else {
            u: train_records.get(u, []) + val_records.get(u, []) for u in train_records}
        m = evaluate(eu, ei, test_records, excl, item_pop)
    print('\n--- BASELINE LightGCN (test) ---')
    for k in TOP_KS:
        print(f'Recall@{k} {m[k]["recall"]:.4f}  NDCG@{k} {m[k]["ndcg"]:.4f}  ARP@{k} {m[k]["arp"]:.4f}')
    return model, adj, item_pop


backbone, sparse_adj, item_pop = train_backbone()

Epoch [1/2000] Loss 0.6809 Recall@50 0.1458 NDCG@50 0.0704 ARP@50 0.5144 3.0s
Epoch [2/2000] Loss 0.5459 Recall@50 0.1443 NDCG@50 0.0696 ARP@50 0.5175 2.0s
Epoch [3/2000] Loss 0.3813 Recall@50 0.1436 NDCG@50 0.0694 ARP@50 0.5174 2.0s
Epoch [4/2000] Loss 0.3273 Recall@50 0.1435 NDCG@50 0.0694 ARP@50 0.5174 2.0s
Epoch [5/2000] Loss 0.3142 Recall@50 0.1436 NDCG@50 0.0694 ARP@50 0.5175 2.1s
Epoch [6/2000] Loss 0.3089 Recall@50 0.1438 NDCG@50 0.0695 ARP@50 0.5177 2.0s
Epoch [7/2000] Loss 0.3068 Recall@50 0.1441 NDCG@50 0.0697 ARP@50 0.5177 2.0s
Epoch [8/2000] Loss 0.3048 Recall@50 0.1437 NDCG@50 0.0695 ARP@50 0.5179 2.1s
Epoch [9/2000] Loss 0.3043 Recall@50 0.1444 NDCG@50 0.0697 ARP@50 0.5180 2.1s
Epoch [10/2000] Loss 0.3018 Recall@50 0.1452 NDCG@50 0.0700 ARP@50 0.5181 2.2s
Epoch [11/2000] Loss 0.2993 Recall@50 0.1447 NDCG@50 0.0699 ARP@50 0.5181 2.1s
Epoch [12/2000] Loss 0.2985 Recall@50 0.1454 NDCG@50 0.0701 ARP@50 0.5182 2.0s
Epoch [13/2000] Loss 0.2974 Recall@50 0.1456 NDCG@50 0.0703 A

In [4]:
# ============================================================================
# CELL 2b — re-score the ALREADY-TRAINED backbone with the fixed metrics.
# No retraining: both fixes live in function definitions and neither affects
# which checkpoint was selected (selection uses NDCG@50, not the tail metric).
# Run this instead of re-running cell 2.
# ============================================================================

@torch.no_grad()
def evaluate(e_u, e_i, eval_recs, exclude_recs, item_pop, ks=TOP_KS):
    users = sorted(u for u, v in eval_recs.items() if v)
    max_k = max(ks)
    ranked_all = []
    for s in range(0, len(users), 512):
        batch = users[s:s + 512]
        scores = e_u[batch] @ e_i.T
        for r, u in enumerate(batch):
            ex = exclude_recs.get(u, [])
            if ex:
                scores[r, torch.tensor(ex, dtype=torch.long, device=scores.device)] = -torch.inf
        ranked_all.append(torch.topk(scores, k=max_k, dim=1).indices.cpu().numpy())
    ranked = np.concatenate(ranked_all, 0)
    truth = [eval_recs[u] for u in users]

    out = {}
    for k in ks:
        r, n = recall_ndcg(truth, ranked, k)
        out[k] = {'recall': r, 'ndcg': n, 'arp': float(item_pop[ranked[:, :k]].mean())}

    # FIX: bottom-20% of items that actually appear in training (notes S15.9).
    # The median split over ALL items swept in the several hundred movies with
    # zero training interactions, which no model can rank.
    present = np.flatnonzero(item_pop > 0)
    cut = np.quantile(item_pop[present], 0.2)
    tail = set(present[item_pop[present] <= cut].tolist())
    tt = [[i for i in t if i in tail] for t in truth]
    keep = [j for j, t in enumerate(tt) if t]
    out['tail_recall@50'] = recall_ndcg([tt[j] for j in keep], ranked[keep], 50)[0] if keep else 0.0
    return out


# FIX: mask training positives only, matching PPAC's test_model and the
# validation call, so val and test share one candidate protocol.
backbone.eval()
with torch.no_grad():
    eu, ei = backbone.readout(sparse_adj)
    m_val  = evaluate(eu, ei, val_records,  train_records, item_pop)
    m_test = evaluate(eu, ei, test_records, train_records, item_pop)

print('--- BASELINE LightGCN (re-scored, fixed metrics) ---')
for tag, m in (('VAL ', m_val), ('TEST', m_test)):
    for k in TOP_KS:
        print(f'{tag} Recall@{k} {m[k]["recall"]:.4f}  NDCG@{k} {m[k]["ndcg"]:.4f}  '
              f'ARP@{k} {m[k]["arp"]:.4f}')
    print(f'{tag} TailRecall@50 {m["tail_recall@50"]:.4f}')

n_present = int((item_pop > 0).sum())
print(f'\n[check] items with training interactions: {n_present}/{len(item_pop)}; '
      f'tail cutoff at 20th pct of those')

--- BASELINE LightGCN (re-scored, fixed metrics) ---
VAL  Recall@20 0.1812  NDCG@20 0.1172  ARP@20 0.3986
VAL  Recall@50 0.3630  NDCG@50 0.1814  ARP@50 0.3334
VAL  Recall@100 0.5564  NDCG@100 0.2376  ARP@100 0.2830
VAL  TailRecall@50 0.0000
TEST Recall@20 0.1886  NDCG@20 0.1464  ARP@20 0.3984
TEST Recall@50 0.3729  NDCG@50 0.2222  ARP@50 0.3332
TEST Recall@100 0.5633  NDCG@100 0.2868  ARP@100 0.2829
TEST TailRecall@50 0.0000

[check] items with training interactions: 3496/3883; tail cutoff at 20th pct of those


In [5]:
print(backbone, len(train_records), item_pop.shape)

LightGCN(
  (user_embedding): Embedding(6038, 64)
  (item_embedding): Embedding(3883, 64)
) 6038 (3883,)


In [ ]:
lens = [len(v) for v in test_records.values()]
items = {i for v in test_records.values() for i in v}
print('users', len(test_records), 'mean pos/user', np.mean(lens))
print('items', len(items), 'interactions', sum(lens))
print('mean train-popularity of test items', item_pop[list(items)].mean())

In [6]:
# ============================================================================
# CELL 3 — STAGE 2: BEHAVIOURAL PROFILES (§7)
# ============================================================================
# Deviation from the old notebook: categories and content vectors come from
# movies.dat genres, NOT from KMeans on the LightGCN item embeddings. Clustering
# the backbone's own embeddings makes q_u a function of the representation being
# debiased -- the profiles then re-encode the popularity signal they are supposed
# to be independent of, and §7 says these come from metadata.
from scipy.stats import spearmanr, entropy
from sklearn.cluster import KMeans

N_CATEGORIES  = 20
CATEGORY_MODE = 'kmeans_genre'   # 'kmeans_genre' | 'primary_genre'
WINDOW_DAYS   = 30               # §7.3 uses *time* windows, not fixed-count chunks
MIN_WINDOW_INTERACTIONS = 3
LOYALTY_WEIGHT = 'count'         # 'count' matches §7.6 with w_ui = #interactions
DECONFOUND = False               # §7.8 -- OFF, as requested
PROXY_COLS = ['diversity', 'temporal_stability', 'exploration', 'cross_category', 'loyalty']


def build_item_metadata(item_map, n_categories=N_CATEGORIES):
    """Genre multi-hot content vectors + one category per item, from movies.dat."""
    genres_of = {}
    vocab = {}
    with open(os.path.join(RAW_DATA_DIR, 'movies.dat'), encoding='latin-1') as fh:
        for line in fh:
            mid, _, gstr = line.strip().split('::')
            gs = gstr.split('|')
            genres_of[item_map[mid]] = gs
            for g in gs:
                vocab.setdefault(g, len(vocab))

    content = np.zeros((len(item_map), len(vocab)), dtype=np.float32)
    for i, gs in genres_of.items():
        for g in gs:
            content[i, vocab[g]] = 1.0
    content /= np.maximum(np.linalg.norm(content, axis=1, keepdims=True), 1e-9)

    if CATEGORY_MODE == 'primary_genre':
        freq = collections.Counter(g for gs in genres_of.values() for g in gs)
        cat = np.array([vocab[min(genres_of[i], key=lambda g: freq[g])]
                        for i in range(len(item_map))], dtype=np.int64)
    else:
        km = KMeans(n_clusters=min(n_categories, len(item_map)), random_state=42, n_init=10)
        cat = km.fit_predict(content).astype(np.int64)
    return content, cat, int(cat.max()) + 1


def user_profiles(interactions, content, category, n_cats):
    """§7.2-7.7. interactions: DataFrame[user, item, rating, timestamp]."""
    recs = []
    for uid, grp in interactions.groupby('user', sort=True):
        items = grp['item'].to_numpy()
        if len(items) < 2:
            continue
        cats = category[items]

        # §7.2 diversity: mean pairwise (1 - cos) over CONTENT vectors
        V = content[items]
        S = V @ V.T
        iu = np.triu_indices(len(items), k=1)
        diversity = float((1.0 - S[iu]).mean())

        # §7.5 cross-category reach: Shannon entropy of P(c|u)
        counts = np.bincount(cats, minlength=n_cats).astype(np.float64)
        probs = counts[counts > 0] / counts.sum()
        cross = float(entropy(probs))

        # §7.4 exploration: fraction of items outside the user's top-3 categories
        top3 = set(np.argsort(-counts)[:3].tolist())
        exploration = float(np.mean([c not in top3 for c in cats]))

        # §7.6 loyalty
        w = (grp['rating'].to_numpy(np.float64) if LOYALTY_WEIGHT == 'rating'
             else np.ones(len(items)))
        agg = np.bincount(pd.factorize(items)[0], weights=w)
        loyalty = float(((agg / agg.sum()) ** 2).sum())

        # §7.3 temporal stability: Spearman between the FULL n_cats-length category
        # histograms of consecutive TIME windows.
        ts = grp['timestamp'].to_numpy(np.int64)
        order = np.argsort(ts)
        ts_o, cats_o = ts[order], cats[order]
        span = WINDOW_DAYS * 86400
        wins, cur, start = [], [], ts_o[0]
        for t, c in zip(ts_o, cats_o):
            if t - start > span and len(cur) >= MIN_WINDOW_INTERACTIONS:
                wins.append(cur); cur, start = [], t
            cur.append(c)
        if len(cur) >= MIN_WINDOW_INTERACTIONS:
            wins.append(cur)
        if len(wins) < 2:                       # fallback: equal-count halves
            half = len(cats_o) // 2
            wins = [cats_o[:half].tolist(), cats_o[half:].tolist()] if half >= 1 else []
        rhos = []
        for a, b in zip(wins[:-1], wins[1:]):
            ha = np.bincount(np.asarray(a), minlength=n_cats).astype(np.float64)
            hb = np.bincount(np.asarray(b), minlength=n_cats).astype(np.float64)
            if ha.std() > 0 and hb.std() > 0:
                rho = spearmanr(ha, hb).statistic
                if not np.isnan(rho):
                    rhos.append(rho)
        temporal = float(np.mean(rhos)) if rhos else 0.0

        recs.append({'user': uid, 'diversity': diversity, 'temporal_stability': temporal,
                     'exploration': exploration, 'cross_category': cross,
                     'loyalty': loyalty, '_degree': len(items)})

    df = pd.DataFrame(recs)
    for c in PROXY_COLS:                                       # §7.7 min-max
        lo, hi = df[c].min(), df[c].max()
        df[c] = (df[c] - lo) / (hi - lo + 1e-8)
    if DECONFOUND:                                             # §7.8 -- OFF
        bins = pd.qcut(df['_degree'], q=10, labels=False, duplicates='drop')
        for c in PROXY_COLS:
            df[c] = df[c] - df.groupby(bins)[c].transform('mean')
    return df.drop(columns=['_degree'])


def build_profiles():
    inter = []
    inv_item = {v: k for k, v in item_map.items()}
    ts_lookup = {}
    with open(os.path.join(RAW_DATA_DIR, 'ratings.dat'), encoding='latin-1') as fh:
        for line in fh:
            u, i, r, t = line.strip().split('::')
            ts_lookup[(int(u), i)] = (float(r), int(t))
    inv_user = {v: k for k, v in user_map.items()}
    for u, items in train_records.items():
        ru = inv_user[u]
        for i in items:
            r, t = ts_lookup[(ru, inv_item[i])]
            inter.append((u, i, r, t))
    df = pd.DataFrame(inter, columns=['user', 'item', 'rating', 'timestamp'])

    content, category, n_cats = build_item_metadata(item_map)
    q_u_df = user_profiles(df, content, category, n_cats)
    print(f'[stage2] q_u for {len(q_u_df)}/{NUM_USERS} users, {n_cats} categories')

    # §7.9 q_i = mean of q_u over N(i); unseen items get the COLUMN MEAN, not 0
    # (0 is the minimum after min-max normalisation, not a neutral value).
    q_i_df = (df[['user', 'item']].merge(q_u_df, on='user', how='inner')
              .groupby('item')[PROXY_COLS].mean().reset_index())
    q_i_df = pd.DataFrame({'item': np.arange(NUM_ITEMS)}).merge(q_i_df, on='item', how='left')
    q_i_df[PROXY_COLS] = q_i_df[PROXY_COLS].fillna(q_i_df[PROXY_COLS].mean())

    q_u = np.tile(q_u_df[PROXY_COLS].mean().to_numpy(np.float32), (NUM_USERS, 1))
    q_u[q_u_df['user'].to_numpy(np.int64)] = q_u_df[PROXY_COLS].to_numpy(np.float32)
    q_i = q_i_df[PROXY_COLS].to_numpy(np.float32)

    q_u_df.to_csv('/kaggle/working/q_u_profiles.csv', index=False)
    q_i_df.to_csv('/kaggle/working/q_i_profiles.csv', index=False)
    return torch.tensor(q_u, device=DEVICE), torch.tensor(q_i, device=DEVICE), df


q_u, q_i, interactions_df = build_profiles()

[stage2] q_u for 6001/6038 users, 20 categories


In [7]:
# ============================================================================
# CELL 4 — STAGE 3: PPD POPULARITY PROXIES (§8)
# ============================================================================
BETA = 0.1
EPS = 1e-8


def ppd_proxies(df, e_u_final, e_i_final, beta=BETA):
    """p_i (§8.1), r_ui (PPD Eq. 6), b_ui = p_i - r_ui (§8.3).

    Fix vs. the old notebook: p_i and r_ui were each min-max normalised on their
    own scale and THEN subtracted, which makes the difference arbitrary. Here the
    difference is taken on the raw scale and normalised once, so b_ui in [0,1]
    is a monotone function of the actual (global - personal) gap. b_ui and
    (1 - b_ui) are used as non-negative centroid weights in §9.3-9.4, so the
    [0,1] mapping is required.
    """
    import scipy.sparse as sp
    u = df['user'].to_numpy(np.int64); i = df['item'].to_numpy(np.int64)
    U = e_u_final.detach().cpu().numpy(); I = e_i_final.detach().cpu().numpy()

    p_raw = U.mean(axis=0) @ I.T                                # §8.1
    R = sp.csr_matrix((np.ones(len(u), np.float32), (u, i)), shape=(len(U), len(I)))
    nbr_sum = R @ I
    sim_sum = (I[i] * nbr_sum[u]).sum(axis=1)
    p_sum = R @ p_raw
    deg = np.asarray(R.sum(axis=1)).ravel()
    r_raw = (sim_sum - beta * p_raw[i] * p_sum[u]) / np.maximum(deg[u], 1.0)   # Eq. 6

    b_raw = p_raw[i] - r_raw                                    # §8.3
    b = (b_raw - b_raw.min()) / (b_raw.max() - b_raw.min() + EPS)

    out = df[['user', 'item']].copy()
    out['p_i'] = p_raw[i]
    out['r_ui'] = r_raw
    out['b_ui'] = b.astype(np.float32)
    return out


with torch.no_grad():
    e_u_final0, e_i_final0 = backbone.readout(sparse_adj)
ppd_df = ppd_proxies(interactions_df, e_u_final0, e_i_final0)
ppd_df.to_csv('/kaggle/working/stage3_ppd_interactions.csv', index=False)
print(f'[stage3] b_ui in [{ppd_df.b_ui.min():.4f}, {ppd_df.b_ui.max():.4f}], '
      f'mean {ppd_df.b_ui.mean():.4f}')

edge_u = torch.tensor(ppd_df['user'].to_numpy(np.int64), device=DEVICE)
edge_i = torch.tensor(ppd_df['item'].to_numpy(np.int64), device=DEVICE)
b_ui = torch.tensor(ppd_df['b_ui'].to_numpy(np.float32), device=DEVICE)

[stage3] b_ui in [0.0000, 1.0000], mean 0.7313


In [8]:
# ============================================================================
# CELL 5 — STAGES 4-7: BEHAVIOUR-GUIDED POPULARITY SUBSPACE
# ============================================================================
NUM_MODES     = 4        # K
PHI           = 0.5      # §9.5
LAMBDA_ORTH   = 1e-2     # §12.5
LAMBDA_REG    = 1e-4
DEBIAS_LR     = 1e-3
DEBIAS_EPOCHS = 500
EVAL_EVERY    = 10
DEBIAS_PATIENCE = 15


class BPSD(nn.Module):
    """Behaviour-guided popularity subspace debiasing.

    Three corrections relative to the earlier implementation:

    1. D is CONSTRUCTED from the centroids on every forward pass (§9.3-9.5,
       §11.2-11.4), not free-trained. Previously the centroids were computed once
       under no_grad as an initialisation and D was then a free nn.Parameter
       optimised by BPR. With the backbone frozen, the only thing BPR can do with
       a free D is rotate it into directions the embeddings barely use, so the
       subtraction removes nothing -- which is exactly what the old run showed
       (removal magnitude 0.111 -> 0.008, ARP@50 0.093 -> 0.386, tail recall
       0.144 -> 0.011). The only trainable parameters are the gates.

    2. No QR. QR returns an orthonormal basis for span(D) whose column k is a
       rotation of the original directions, so g_{v,k} would be gating a basis
       vector that no longer corresponds to mode k. Instead each d_k is scaled to
       unit norm (which is all §9.10 needs) and L_orth pushes the modes apart.

    3. D^u and D^i are kept separate -- the Normalize(d_k^u + d_k^i) step of
       §11.4 is skipped, as requested, so §9.5's D_u / D_i are used directly.
    """

    def __init__(self, n_proxies=5, num_modes=NUM_MODES, phi=PHI, uniform_gates=False):
        super().__init__()
        self.K, self.phi = num_modes, phi
        # uniform_gates=True freezes g at 1/K (notes S15.3). Use it for the
        # multi-directionality control: K=1 has NO trainable parameters at all,
        # because softmax over a single logit is identically 1.0 and the gate
        # layers get zero gradient. Comparing trainable K=4 against K=1 would
        # therefore conflate "more directions" with "has parameters". Running
        # K=4 with uniform gates against K=1 isolates the dimensionality claim.
        self.uniform_gates = uniform_gates
        self.user_gate = nn.Linear(n_proxies, num_modes)   # W_u, b_u
        self.item_gate = nn.Linear(n_proxies, num_modes)   # W_i, b_i
        for lin in (self.user_gate, self.item_gate):
            nn.init.xavier_uniform_(lin.weight); nn.init.zeros_(lin.bias)

    def has_trainable_gates(self):
        return (not self.uniform_gates) and self.K > 1

    def gates(self, q_u, q_i):                             # §9.1
        if self.uniform_gates:
            return (q_u.new_full((q_u.shape[0], self.K), 1.0 / self.K),
                    q_i.new_full((q_i.shape[0], self.K), 1.0 / self.K))
        return (torch.softmax(self.user_gate(q_u), dim=-1),
                torch.softmax(self.item_gate(q_i), dim=-1))

    @staticmethod
    def _centroid(node_idx, edge_w, gate_rows, values, n_nodes):
        """sum_{(u,i)} w * g_k * e  /  sum_{(u,i)} w * g_k, done without
        materialising an (|E|, d) tensor: aggregate the scalar coefficients per
        node first, then one (K, n_nodes) @ (n_nodes, d) matmul."""
        coef = torch.zeros(n_nodes, gate_rows.shape[1],
                           device=values.device, dtype=values.dtype)
        coef = coef.index_add(0, node_idx, edge_w.unsqueeze(1) * gate_rows)
        return (coef.t() @ values) / (coef.sum(dim=0).unsqueeze(1) + EPS)

    def directions(self, e_u0, e_i0, g_u, g_i, edge_u, edge_i, b):
        # user side: centroids over neighbour ITEM embeddings, gated by g_u
        c_pop_u = self._centroid(edge_i, b, g_u[edge_u], e_i0, e_i0.shape[0])
        c_pre_u = self._centroid(edge_i, 1.0 - b, g_u[edge_u], e_i0, e_i0.shape[0])
        # item side: centroids over neighbour USER embeddings, gated by g_i
        c_pop_i = self._centroid(edge_u, b, g_i[edge_i], e_u0, e_u0.shape[0])
        c_pre_i = self._centroid(edge_u, 1.0 - b, g_i[edge_i], e_u0, e_u0.shape[0])
        D_u = c_pop_u - self.phi * c_pre_u                  # §9.5, (K, d)
        D_i = c_pop_i - self.phi * c_pre_i
        return D_u, D_i

    @staticmethod
    def _orth_penalty(D):                                   # §11.5
        Dn = F.normalize(D, dim=1)
        I = torch.eye(D.shape[0], device=D.device, dtype=D.dtype)
        return (Dn @ Dn.t() - I).pow(2).sum()

    def forward(self, e_u0, e_i0, q_u, q_i, edge_u, edge_i, b):
        g_u, g_i = self.gates(q_u, q_i)
        D_u, D_i = self.directions(e_u0, e_i0, g_u, g_i, edge_u, edge_i, b)
        orth = self._orth_penalty(D_u) + self._orth_penalty(D_i)

        Du = F.normalize(D_u, dim=1)                        # unit-norm d_k
        Di = F.normalize(D_i, dim=1)
        # §9.10:  e~ = e - sum_k g_k (d_k^T e) d_k
        rm_u = ((e_u0 @ Du.t()) * g_u) @ Du
        rm_i = ((e_i0 @ Di.t()) * g_i) @ Di
        return e_u0 - rm_u, e_i0 - rm_i, g_u, g_i, orth, (rm_u, rm_i, Du, Di)


def train_debiaser(num_modes=NUM_MODES, verbose=True, uniform_gates=False):
    set_seed(SEED)
    e_u_frozen = backbone.user_embedding.weight.detach().to(DEVICE)   # §13.2 frozen
    e_i_frozen = backbone.item_embedding.weight.detach().to(DEVICE)

    model = BPSD(n_proxies=len(PROXY_COLS), num_modes=num_modes,
                 uniform_gates=uniform_gates).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=DEBIAS_LR)
    tag = f'K{num_modes}' + ('_uniform' if uniform_gates else '')
    ckpt = f'/kaggle/working/bpsd_{tag}.pt'
    best, best_ep, stall = -np.inf, 0, 0

    # A parameter-free variant is fully determined by the frozen embeddings and
    # b_ui, so every epoch would give an identical result. Evaluate once.
    n_epochs = DEBIAS_EPOCHS if model.has_trainable_gates() else EVAL_EVERY
    if not model.has_trainable_gates():
        print(f'[{tag}] gates carry no gradient; evaluating the fixed projection once.')

    for epoch in range(1, n_epochs + 1):
        model.train(); t0 = time.time()
        u, p, n = sample_triplets(train_records, NUM_ITEMS)
        perm = torch.randperm(len(u)); u, p, n = u[perm].to(DEVICE), p[perm].to(DEVICE), n[perm].to(DEVICE)

        opt.zero_grad(set_to_none=True)
        eu0, ei0, g_u_, g_i_, orth, (rm_u, rm_i, Du, Di) = model(
            e_u_frozen, e_i_frozen, q_u, q_i, edge_u, edge_i, b_ui)
        eu, ei = propagate(sparse_adj, eu0, ei0)

        bpr = torch.zeros((), device=DEVICE)
        for s in range(0, len(u), BATCH_SIZE):
            sl = slice(s, s + BATCH_SIZE)
            m = (eu[u[sl]] * ei[p[sl]]).sum(1) - (eu[u[sl]] * ei[n[sl]]).sum(1)
            bpr = bpr + F.softplus(-m).sum()
        bpr = bpr / len(u)

        reg = sum(prm.pow(2).sum() for prm in model.parameters())   # §12.5 L_reg
        loss = bpr + LAMBDA_ORTH * orth + LAMBDA_REG * reg
        loss.backward(); opt.step()

        if epoch % EVAL_EVERY:
            continue
        model.eval()
        with torch.no_grad():
            eu0, ei0, *_ = model(e_u_frozen, e_i_frozen, q_u, q_i, edge_u, edge_i, b_ui)
            eu, ei = propagate(sparse_adj, eu0, ei0)
            m = evaluate(eu, ei, val_records, train_records, item_pop)
        if m[50]['ndcg'] > best:
            best, best_ep, stall = m[50]['ndcg'], epoch, 0
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, ckpt)
        else:
            stall += 1
        if verbose:
            with torch.no_grad():
                cos = Du @ Du.t()
                offdiag = (cos - torch.eye(num_modes, device=DEVICE)).abs().max().item()
                gate_spread = g_u_.std(dim=0).mean().item()
            print(f'Epoch [{epoch}/{DEBIAS_EPOCHS}] loss {loss.item():.5f} '
                  f'(BPR {bpr.item():.5f}, orth {orth.item():.5f}) '
                  f'removed |u| {rm_u.norm(dim=1).mean().item():.4f} '
                  f'maxcos {offdiag:.3f} gatestd {gate_spread:.3f} '
                  f'Recall@50 {m[50]["recall"]:.4f} NDCG@50 {m[50]["ndcg"]:.4f} '
                  f'ARP@50 {m[50]["arp"]:.4f} tail {m["tail_recall@50"]:.4f} '
                  f'{time.time()-t0:.1f}s')
        if stall >= DEBIAS_PATIENCE:
            print(f'Early stop at {epoch}; best {best_ep}'); break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)['model_state_dict'])
    model.eval()
    with torch.no_grad():
        eu0, ei0, *_ = model(e_u_frozen, e_i_frozen, q_u, q_i, edge_u, edge_i, b_ui)
        eu, ei = propagate(sparse_adj, eu0, ei0)
        m = evaluate(eu, ei, test_records, train_records, item_pop)
    print(f'\n--- BPSD ({tag}) TEST ---')
    for k in TOP_KS:
        print(f'Recall@{k} {m[k]["recall"]:.4f}  NDCG@{k} {m[k]["ndcg"]:.4f}  ARP@{k} {m[k]["arp"]:.4f}')
    print(f'TailRecall@50 {m["tail_recall@50"]:.4f}')
    return model, m


# Full model, learned behavioural gating.
bpsd_k4,  m_k4  = train_debiaser(4)

# Controls. K=1 is parameter-free by construction; K=4 with uniform gates is the
# matched comparison that isolates multi-directionality from learned gating.
bpsd_k1,  m_k1  = train_debiaser(1)
bpsd_k4u, m_k4u = train_debiaser(4, uniform_gates=True)

print('\n=== SUMMARY (test) ===')
for name, m in (('K=1 (single direction)', m_k1),
                ('K=4 uniform gates', m_k4u),
                ('K=4 learned gates', m_k4)):
    print(f'{name:24s} Recall@50 {m[50]["recall"]:.4f}  NDCG@50 {m[50]["ndcg"]:.4f}  '
          f'ARP@50 {m[50]["arp"]:.4f}  Tail@50 {m["tail_recall@50"]:.4f}')

# §15.2 -- full K sweep once the three-way comparison looks sensible.
# results = {K: train_debiaser(K, verbose=False)[1] for K in [1, 2, 4, 8, 16]}

Epoch [10/500] loss 0.54989 (BPR 0.30926, orth 23.99476) removed |u| 2.5369 maxcos 1.000 gatestd 0.029 Recall@50 0.1412 NDCG@50 0.0704 ARP@50 0.0908 tail 0.0000 0.8s
Epoch [20/500] loss 0.54832 (BPR 0.30773, orth 23.99493) removed |u| 2.5369 maxcos 1.000 gatestd 0.029 Recall@50 0.1412 NDCG@50 0.0704 ARP@50 0.0908 tail 0.0000 0.9s
Epoch [30/500] loss 0.54926 (BPR 0.30868, orth 23.99511) removed |u| 2.5368 maxcos 1.000 gatestd 0.028 Recall@50 0.1412 NDCG@50 0.0704 ARP@50 0.0908 tail 0.0000 0.7s
Epoch [40/500] loss 0.54899 (BPR 0.30844, orth 23.99528) removed |u| 2.5368 maxcos 1.000 gatestd 0.028 Recall@50 0.1412 NDCG@50 0.0704 ARP@50 0.0908 tail 0.0000 0.7s
Epoch [50/500] loss 0.54900 (BPR 0.30847, orth 23.99544) removed |u| 2.5368 maxcos 1.000 gatestd 0.027 Recall@50 0.1412 NDCG@50 0.0704 ARP@50 0.0908 tail 0.0000 0.7s
Epoch [60/500] loss 0.54942 (BPR 0.30891, orth 23.99561) removed |u| 2.5368 maxcos 1.000 gatestd 0.027 Recall@50 0.1412 NDCG@50 0.0704 ARP@50 0.0908 tail 0.0000 0.7s
Epoc

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# DIAGNOSTIC — is PHI the cause of the mode collapse?
# Run this after cell 5 has defined BPSD. It trains nothing.
# ============================================================================
e_u_frozen = backbone.user_embedding.weight.detach().to(DEVICE)
e_i_frozen = backbone.item_embedding.weight.detach().to(DEVICE)
m_item = F.normalize(e_i_frozen.mean(0), dim=0)          # global mean item vector

print(f'mean ||e_u|| = {e_u_frozen.norm(dim=1).mean():.4f}   '
      f'mean ||e_i|| = {e_i_frozen.norm(dim=1).mean():.4f}')
print(f'{"phi":>5} {"maxcos(d_j,d_k)":>16} {"|cos(d_1, mean_item)|":>22} {"pred removed |u|":>18}')

with torch.no_grad():
    probe = BPSD(n_proxies=len(PROXY_COLS), num_modes=4).to(DEVICE)
    g_u, g_i = probe.gates(q_u, q_i)
    for phi in [0.0, 0.5, 0.9, 1.0]:
        probe.phi = phi
        D_u, _ = probe.directions(e_u_frozen, e_i_frozen, g_u, g_i, edge_u, edge_i, b_ui)
        Du = F.normalize(D_u, dim=1)
        cos = Du @ Du.t()
        off = (cos - torch.eye(4, device=DEVICE)).abs().max().item()
        align = (Du[0] @ m_item).abs().item()
        rm = ((e_u_frozen @ Du.t()) * g_u) @ Du
        print(f'{phi:>5.2f} {off:>16.4f} {align:>22.4f} {rm.norm(dim=1).mean().item():>18.4f}')

print('\nmaxcos near 1.0 => modes collapsed. High |cos(d_1, mean_item)| => the')
print('direction is just the global mean vector, so removing it destroys signal.')



# Output - 
# ```
# mean ||e_u|| = 7.7863   mean ||e_i|| = 7.5937
#   phi  maxcos(d_j,d_k)  |cos(d_1, mean_item)|   pred removed |u|
#  0.00           0.9999                 0.8306             2.4308
#  0.50           1.0000                 0.8719             2.5368
#  0.90           1.0000                 0.9021             2.6096
#  1.00           1.0000                 0.9081             2.6225

# maxcos near 1.0 => modes collapsed. High |cos(d_1, mean_item)| => the
# direction is just the global mean vector, so removing it destroys signal
# ```

In [ ]:
# ============================================================================
# DIAGNOSTIC 2 — what actually breaks the mode collapse?
#
# Root cause established: g_{u,k} ~ 1/K makes the 1/K factor cancel out of the
# centroid ratio, so c_pop,k is the same b-weighted mean item vector for every k.
#
# Two independent levers:
#   (a) gate sharpness   -- soft/temperature vs hard one-hot assignment.
#       Sharp gates give each mode a DIFFERENT user group, hence a different
#       item distribution, hence a genuinely different centroid.
#   (b) centroid scaling -- normalise c_pop and c_pref to unit norm BEFORE
#       differencing, so d_k = pop_direction - phi*pref_direction cancels the
#       shared mean instead of leaving a magnitude difference along it.
#
# Trains nothing. Run after cell 5 defines BPSD.
# ============================================================================
e_u_frozen = backbone.user_embedding.weight.detach().to(DEVICE)
e_i_frozen = backbone.item_embedding.weight.detach().to(DEVICE)
m_item = F.normalize(e_i_frozen.mean(0), dim=0)
K = 4


def centroid(node_idx, w, gate_rows, values, n_nodes):
    coef = torch.zeros(n_nodes, gate_rows.shape[1], device=values.device, dtype=values.dtype)
    coef = coef.index_add(0, node_idx, w.unsqueeze(1) * gate_rows)
    return (coef.t() @ values) / (coef.sum(0).unsqueeze(1) + 1e-8)


def build_dirs(g_u, phi, unit_centroids):
    c_pop = centroid(edge_i, b_ui, g_u[edge_u], e_i_frozen, e_i_frozen.shape[0])
    c_pre = centroid(edge_i, 1.0 - b_ui, g_u[edge_u], e_i_frozen, e_i_frozen.shape[0])
    if unit_centroids:
        c_pop, c_pre = F.normalize(c_pop, dim=1), F.normalize(c_pre, dim=1)
    return c_pop - phi * c_pre


with torch.no_grad():
    probe = BPSD(n_proxies=len(PROXY_COLS), num_modes=K).to(DEVICE)
    logits_u = probe.user_gate(q_u)

    print(f'{"gates":>14} {"unit_c":>7} {"phi":>5} {"maxcos":>8} '
          f'{"|cos(d1,mean)|":>15} {"removed|u|":>11} {"%of||e_u||":>11}')
    norm_u = e_u_frozen.norm(dim=1).mean().item()

    for label, g in [('soft T=1.0',   torch.softmax(logits_u / 1.0,  -1)),
                     ('soft T=0.2',   torch.softmax(logits_u / 0.2,  -1)),
                     ('soft T=0.05',  torch.softmax(logits_u / 0.05, -1)),
                     ('hard argmax',  F.one_hot(logits_u.argmax(-1), K).float()),
                     ('random hard',  F.one_hot(torch.randint(0, K, (q_u.shape[0],),
                                                device=DEVICE), K).float())]:
        for unit_c in (False, True):
            for phi in (0.5, 1.0):
                D = build_dirs(g, phi, unit_c)
                Dn = F.normalize(D, dim=1)
                off = (Dn @ Dn.t() - torch.eye(K, device=DEVICE)).abs().max().item()
                align = (Dn[0] @ m_item).abs().item()
                rm = ((e_u_frozen @ Dn.t()) * g) @ Dn
                r = rm.norm(dim=1).mean().item()
                print(f'{label:>14} {str(unit_c):>7} {phi:>5.1f} {off:>8.4f} '
                      f'{align:>15.4f} {r:>11.4f} {100*r/norm_u:>10.1f}%')

    frac = F.one_hot(logits_u.argmax(-1), K).float().mean(0)
    print(f'\nhard-assignment mode occupancy: {[round(x, 3) for x in frac.tolist()]}')
    print('(all mass in one mode => the gate net is not separating users at all)')
    
    
#     Output - 
#     ```
#           gates  unit_c   phi   maxcos  |cos(d1,mean)|  removed|u|  %of||e_u||
#     soft T=1.0   False   0.5   1.0000          0.8735      2.5377       32.6%
#     soft T=1.0   False   1.0   1.0000          0.9103      2.6225       33.7%
#     soft T=1.0    True   0.5   0.9998          0.9198      2.6314       33.8%
#     soft T=1.0    True   1.0   0.9981          0.6554      1.8444       23.7%
#     soft T=0.2   False   0.5   0.9965          0.8799      2.5545       32.8%
#     soft T=0.2   False   1.0   0.9979          0.9205      2.6236       33.7%
#     soft T=0.2    True   0.5   0.9969          0.9159      2.6351       33.8%
#     soft T=0.2    True   1.0   0.9897          0.9130      1.8610       23.9%
#    soft T=0.05   False   0.5   0.9461          0.8018      2.5603       32.9%
#    soft T=0.05   False   1.0   0.9918          0.9131      2.6295       33.8%
#    soft T=0.05    True   0.5   0.9330          0.7726      2.6326       33.8%
#    soft T=0.05    True   1.0   0.9873          0.9046      1.9416       24.9%
#    hard argmax   False   0.5   1.0000          0.0000      2.5652       32.9%
#    hard argmax   False   1.0   1.0000          0.0000      2.6311       33.8%
#    hard argmax    True   0.5   1.0000          0.0000      2.6439       34.0%
#    hard argmax    True   1.0   1.0000          0.0000      1.9741       25.4%
#    random hard   False   0.5   0.9984          0.8778      2.5366       32.6%
#    random hard   False   1.0   0.9990          0.9114      2.6216       33.7%
#    random hard    True   0.5   0.9986          0.9196      2.6346       33.8%
#    random hard    True   1.0   0.9889          0.6523      1.8442       23.7%

# hard-assignment mode occupancy: [0.0, 0.822, 0.001, 0.177]
# (all mass in one mode => the gate net is not separating users at all
# ```

In [11]:
# ============================================================================
# DIAGNOSTIC 3 — can ANY construction give K genuinely distinct directions?
#
# Established: c_pop,k is a mean over ~360k edges, so every user grouping
# (behavioural, random, 82/18 hard) yields the same b-weighted item mean.
# Conditioning a mean on a user partition cannot separate the modes.
#
# A: centred residuals   d_k = (c_pop,k - c_bar_pop) - phi(c_pref,k - c_bar_pref)
#    The across-mode mean IS PPD's single global direction; modes become
#    behaviour-conditioned corrections around it.
# B: principal axes      top-K PCs of the popularity-weighted item scatter.
#    Orthogonal by construction; gates then weight how much of each axis to remove.
#
# Trains nothing.
# ============================================================================
e_u_frozen = backbone.user_embedding.weight.detach().to(DEVICE)
e_i_frozen = backbone.item_embedding.weight.detach().to(DEVICE)
m_item = F.normalize(e_i_frozen.mean(0), dim=0)
norm_u = e_u_frozen.norm(dim=1).mean().item()
K = 4


def centroid(node_idx, w, gate_rows, values, n_nodes):
    coef = torch.zeros(n_nodes, gate_rows.shape[1], device=values.device, dtype=values.dtype)
    coef = coef.index_add(0, node_idx, w.unsqueeze(1) * gate_rows)
    occ = coef.sum(0)
    return (coef.t() @ values) / (occ.unsqueeze(1) + 1e-8), occ


def report(name, D, g):
    Dn = F.normalize(D, dim=1)
    off = (Dn @ Dn.t() - torch.eye(Dn.shape[0], device=DEVICE)).abs().max().item()
    align = (Dn @ m_item).abs().max().item()
    rm = ((e_u_frozen @ Dn.t()) * g) @ Dn
    r = rm.norm(dim=1).mean().item()
    print(f'{name:>34} {off:>8.4f} {align:>15.4f} {r:>11.4f} {100*r/norm_u:>10.1f}%')


with torch.no_grad():
    probe = BPSD(n_proxies=len(PROXY_COLS), num_modes=K).to(DEVICE)
    logits_u = probe.user_gate(q_u)
    g_soft = torch.softmax(logits_u, -1)
    g_unif = torch.full_like(g_soft, 1.0 / K)

    print(f'{"construction":>34} {"maxcos":>8} {"max|cos(d,mean)|":>15} '
          f'{"removed|u|":>11} {"%of||e_u||":>11}')

    # ---- baseline: as currently implemented -------------------------------
    c_pop, _ = centroid(edge_i, b_ui, g_soft[edge_u], e_i_frozen, e_i_frozen.shape[0])
    c_pre, _ = centroid(edge_i, 1.0 - b_ui, g_soft[edge_u], e_i_frozen, e_i_frozen.shape[0])
    report('current (phi=1.0)', c_pop - c_pre, g_soft)

    # ---- A: centred residuals ---------------------------------------------
    for phi in (0.5, 1.0):
        D = (c_pop - c_pop.mean(0, keepdim=True)) - phi * (c_pre - c_pre.mean(0, keepdim=True))
        report(f'A centred residual (phi={phi})', D, g_soft)

    # A with unit-normalised centroids first
    cp, cr = F.normalize(c_pop, dim=1), F.normalize(c_pre, dim=1)
    D = (cp - cp.mean(0, keepdim=True)) - 1.0 * (cr - cr.mean(0, keepdim=True))
    report('A centred + unit centroids', D, g_soft)

    # ---- B: principal axes of the popularity-weighted item scatter --------
    w_item = torch.zeros(e_i_frozen.shape[0], device=DEVICE)
    w_item = w_item.index_add(0, edge_i, b_ui)
    w = (w_item / w_item.sum()).unsqueeze(1)
    mu = (w * e_i_frozen).sum(0, keepdim=True)
    Xc = (e_i_frozen - mu) * w.sqrt()
    _, S, V = torch.linalg.svd(Xc, full_matrices=False)
    report('B top-K popularity PCs', V[:K], g_soft)
    report('B top-K PCs, uniform gates', V[:K], g_unif)
    print(f'   PC singular values: {[round(x, 3) for x in S[:K + 2].tolist()]}')

    # ---- reference: what does the global (PPD-style) direction cost? ------
    cg_pop, _ = centroid(edge_i, b_ui, torch.ones_like(b_ui).unsqueeze(1),
                         e_i_frozen, e_i_frozen.shape[0])
    cg_pre, _ = centroid(edge_i, 1.0 - b_ui, torch.ones_like(b_ui).unsqueeze(1),
                         e_i_frozen, e_i_frozen.shape[0])
    report('PPD single global direction', cg_pop - cg_pre, torch.ones(q_u.shape[0], 1, device=DEVICE))

print('\nWant: maxcos well below 0.5, max|cos(d,mean)| low, removed% modest (<15%).')

                      construction   maxcos max|cos(d,mean)|  removed|u|  %of||e_u||
                 current (phi=1.0)   1.0000          0.9136      2.6227       33.7%
      A centred residual (phi=0.5)   0.9563          0.7295      1.6267       20.9%
      A centred residual (phi=1.0)   0.9933          0.9136      2.2999       29.5%
        A centred + unit centroids   0.9963          0.8571      2.2302       28.6%
            B top-K popularity PCs   0.0000          0.9352      1.2840       16.5%
        B top-K PCs, uniform gates   0.0000          0.9352      1.1139       14.3%
   PC singular values: [2.494, 1.512, 1.4, 1.37, 1.34, 1.322]
       PPD single global direction   0.0000          0.9088      2.6226       33.7%

Want: maxcos well below 0.5, max|cos(d,mean)| low, removed% modest (<15%).
